In [ ]:
# 如果没装requests
# !pip install requests

import requests
import json
import numpy as np

# ========== 配置部分 调用LLM接口 ==========

API_BASE_URL = " "  # (第三方)代理地址
API_KEY = " "                # <<< 记得换成自己的key
MODEL_NAME = "gpt-4o"                     # 模型选择

# ========== 辅助函数 ==========

def clean_response(text):
    """清理GPT输出"""
    text = text.replace('```json', '').replace('```', '').strip()
    return text

def post_to_proxy(prompt, model=MODEL_NAME, temperature=0.7, max_retries=3):
    """请求代理API，增加编码修正+异常重试"""
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}"
    }
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature
    }
    url = f"{API_BASE_URL}/chat/completions"

    for attempt in range(max_retries):
        try:
            response = requests.post(url, headers=headers, data=json.dumps(payload, ensure_ascii=False).encode('utf-8'))
            if response.status_code == 200:
                return response.json()
            else:
                print(f"⚠️ 第{attempt+1}次请求失败，状态码{response.status_code}: {response.text}")
        except Exception as e:
            print(f"⚠️ 第{attempt+1}次请求异常: {e}")

    raise Exception(f"❌ 在尝试{max_retries}次后仍无法完成请求！")

# ========== System Model专用Prompt模板 ==========

factor_role_instrutor = """
ROLE INSTRUCTION: You are good at understanding tasks and writing python codes.
You should fully understand the provided task and describe the exact observation form in the current system model.
Then, based on your understanding, analyze potential positive and negative behaviours or statuses that can be reflected in the observation.
Finally, write an evaluation function that returns factors evaluating the current system status from different aspects.

Note:
1. Do not use information you are not given!
2. Focus on the most relevant evaluation factors and use information in observation as little as possible.
3. The code should be generic, complete and not contain omissions!
4. Avoid dividing by zero!
5. The input variable is in the form of (batch_size, dim), please return a list of several evaluation factor arrays, each in the form of (batch_size, 1).

Please think step by step and adhere to the following JSON format:
{
"Understand": "(your thought about the system model and observation)",
"Analyze": "(your step-by-step analysis of potential good/bad statuses)",
"Functions": "(a python function like 'def evaluation_func(observation): ...')"
}
"""

# ========== Prompt封装类 ==========

class Base_prompt(object):
    def __init__(self, map_name):
        self.map_name = map_name
        self.task_description = ''
        self.state_form = ''
        self.role_instruction = factor_role_instrutor

    def get_prompt(self):
        return self.task_description + self.state_form + self.role_instruction

    def factor_check(self, content_json, n_agents):
        """验证evaluation function执行正确"""
        error_content = ''
        try:
            func = content_json['Functions']
            namespace = {}
            exec(func, namespace)
            active_evaluation_func = namespace['evaluation_func']
            dummy_obs = np.ones((n_agents, 10))
            evaluation_factors = active_evaluation_func(dummy_obs)
            for factor in evaluation_factors:
                if factor.shape != (n_agents, 1):
                    raise ValueError("输出shape不正确，应为(batch_size, 1)")
            return True, active_evaluation_func, len(evaluation_factors)
        except Exception as e:
            error_content = str(e)
            return False, None, error_content

# ========== 打分系统 ==========

def score_reward_function(eval_func, n_agents):
    """根据reward函数的输出特性打分"""
    dummy_obs = np.random.randn(n_agents * 10, 10)
    evaluation_factors = eval_func(dummy_obs)

    scores = []
    for factor in evaluation_factors:
        if factor.shape != (dummy_obs.shape[0], 1):
            continue
        var = np.var(factor)
        mean_abs = np.mean(np.abs(factor))
        score = var / (mean_abs + 1e-6)
        scores.append(score)

    final_score = np.mean(scores)
    return final_score

# ========== 生成+筛选+打分模块 ==========

def generate_and_select_best_reward(env_name, map_name, task_description, state_form, n_agents,
                                    model=MODEL_NAME, candidate_num=5, retries_max=5):
    """生成多个奖励函数，自动打分，选最高的"""
    base_prompt = Base_prompt(map_name)
    base_prompt.task_description = task_description
    base_prompt.state_form = state_form

    prompt_text = base_prompt.get_prompt()

    valid_candidates = []
    attempts = 0

    print(f"🚀 开始生成 {candidate_num} 个Latent Reward候选...")

    while len(valid_candidates) < candidate_num and attempts < candidate_num * retries_max:
        attempts += 1

        print(f"\n🎲 正在尝试第 {attempts} 个候选函数...")

        try:
            response = post_to_proxy(prompt_text, model=model)
            gpt_output = response['choices'][0]['message']['content']
            gpt_output_cleaned = clean_response(gpt_output)
            parsed_json = json.loads(gpt_output_cleaned)

            valid, eval_func, info = base_prompt.factor_check(parsed_json, n_agents)
            if valid:
                score = score_reward_function(eval_func, n_agents)
                valid_candidates.append((parsed_json, score))
                print(f"✅ 第{len(valid_candidates)}个合格！Score: {score:.6f}")
            else:
                print(f"⚠️ 验证失败：{info}")

        except Exception as e:
            print(f"⚠️ 推理或解析失败: {e}")

    if not valid_candidates:
        return None, "❌ 未能生成任何合格Latent Reward函数！"

    valid_candidates.sort(key=lambda x: x[1], reverse=True)

    print(f"\n🎯 最佳Latent Reward函数得分: {valid_candidates[0][1]:.6f}")

    return valid_candidates[0][0]  # 返回得分最高的那个

# ========== 新增：根据Latent Factors生成最终Reward Function ==========

def generate_final_reward_function(factor_list_descriptions, model=MODEL_NAME):
    prompt = f"""
你已经拥有如下Latent Reward因子：
{factor_list_descriptions}
请基于这些因子，设计一个综合奖励函数reward_function。
要求：
- 输入是一个list，包含这些因子（每个因子是(batch_size,1)数组）
- 输出是(batch_size,1)的奖励数组
- 奖励应最大化能量效率、数据传输成功率，最小化AoI等，保证奖励稳定且连续。
- 请直接给出标准Python代码，格式为：

def reward_function(factors):
    ...
    return final_reward
"""
    response = post_to_proxy(prompt, model=model)
    gpt_output = response['choices'][0]['message']['content']
    gpt_output_cleaned = clean_response(gpt_output)
    print("\n🏆 生成的最终Reward Function:")
    print(gpt_output_cleaned)
    return gpt_output_cleaned

# ========== 示例使用 ==========

if __name__ == "__main__":
    env_name = 'SystemModel'
    map_name = 'UAV-assisted MIoT'

    task_description = """
    In this system, a UAV assists a maritime Internet of Things (MIoT) network.
    The UAV flies to an optimal hovering position, transfers wireless energy to Marine IoT Terminals (MITs),
    collects data from them, and then offloads it to a Mobile Base Station (MBS).
    Objectives: Minimize total energy consumption, ensure successful data transmission, and maintain data freshness (AoI).
    """

    state_form = """
    Each MIT observation contains:
    - (Xn, Yn) position
    - Residual energy
    - Distance to UAV
    - Channel gain to UAV
    - Remaining data size to upload
    - Current Age of Information (AoI)

    UAV observation contains:
    - (X(t), Y(t)) current position
    - Distance to MBS
    - Channel gain to MBS
    """

    n_agents = 5  # 假设有5个MIT终端

    best_result = generate_and_select_best_reward(
        env_name, map_name, task_description, state_form, n_agents,
        model="gpt-4o", candidate_num=10
    )

    if best_result:
        print("\n🏆 最优Latent Reward函数：")
        print(json.dumps(best_result, indent=2, ensure_ascii=False))

        # 解析Latent Reward因子名称（这里假设根据你的系统，大概如下）
        factor_list_description = """
        1. energy_efficiency: Channel gain over distance to UAV
        2. data_transmission_efficiency: Channel gain over remaining data size
        3. data_freshness: Inverse of AoI
        4. UAV_comm_efficiency: UAV to MBS communication efficiency
        """

        generate_final_reward_function(factor_list_description)

    else:
        print("❌ 错误，未生成成功！")
